In [ ]:
import sys
!{sys.executable} -m pip install pandas

  Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 8.5 MB/s eta 0:00:02
   ----- ---------------------------------- 1.3/9.7 MB 3.7 MB/s eta 0:00:03
   --------- ------------------------------ 2.4/9.7 MB 4.5 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.7 MB 4.3 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.7 MB 4.2 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.7 MB 4.1 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.7 MB 4.1 MB/s eta 0:00:02
   ------------------------- -------------- 6.3/9.7 MB 4.1 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 4.0 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.7 MB 4.0 MB/s eta 0:00:01
   -----------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
df=pd.read_csv("C:\\Users\\Bhairavi\\OneDrive\\GUVI- DSA\\Projects\\Project 2 - Luxury Housing sales\\Luxury_Housing_Bangalore.csv")

In [2]:
list(df.columns)  ## List of Columns

['Property_ID',
 'Micro_Market',
 'Project_Name',
 'Developer_Name',
 'Unit_Size_Sqft',
 'Configuration',
 'Ticket_Price_Cr',
 'Transaction_Type',
 'Buyer_Type',
 'Purchase_Quarter',
 'Connectivity_Score',
 'Amenity_Score',
 'Possession_Status',
 'Sales_Channel',
 'NRI_Buyer',
 'Locality_Infra_Score',
 'Avg_Traffic_Time_Min',
 'Buyer_Comments']

In [3]:
df['Micro_Market'].value_counts  ##checking the count and datatype

<bound method IndexOpsMixin.value_counts of 0             Sarjapur Road
1               Indiranagar
2         Bannerghatta Road
3              bellary road
4               Koramangala
                ...        
100995         BELLARY ROAD
100996         Bellary Road
100997          HENNUR ROAD
100998          rajajinagar
100999           whitefield
Name: Micro_Market, Length: 101000, dtype: str>

In [4]:
df["Micro_Market"] = (  ## normalizing the texts
    df["Micro_Market"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.title()
)

In [5]:
df["Ticket_Price_Cr"] = (    ## formatting numbers
    df["Ticket_Price_Cr"]
        .astype(str)
        .str.replace("₹", "", regex=False)
        .str.replace("Cr", "", regex=False)
        .str.replace("cr", "", regex=False)
        .str.strip()
)

df["Ticket_Price_Cr"] = pd.to_numeric(df["Ticket_Price_Cr"], errors="coerce")

In [6]:
df[["Unit_Size_Sqft","Ticket_Price_Cr"]].isna().sum()  ## Find nulls

Unit_Size_Sqft     10046
Ticket_Price_Cr    10019
dtype: int64

In [7]:
df[
    df["Unit_Size_Sqft"].isna() & df["Ticket_Price_Cr"].isna()
].shape[0]

949

In [8]:
before = len(df)  ## dropping null

df = df.dropna(
    subset=["Unit_Size_Sqft", "Ticket_Price_Cr"],
    how="all"
)

after = len(df)
print(before - after)


949


In [9]:
df = df.dropna(subset=["Unit_Size_Sqft", "Ticket_Price_Cr"])

In [10]:
df[["Unit_Size_Sqft","Ticket_Price_Cr"]].isna().sum()

Unit_Size_Sqft     0
Ticket_Price_Cr    0
dtype: int64

In [11]:
df["Amenity_Score"] = df["Amenity_Score"].fillna(
    df["Amenity_Score"].median()
)

In [12]:
df['Amenity_Score'].isna().sum()

np.int64(0)

In [13]:
df["Buyer_Comments"] = df["Buyer_Comments"].fillna("No comments")

In [14]:
df['Amenity_Score'].isna().sum()

np.int64(0)

In [15]:
df["Developer_Name"] = df["Developer_Name"].where(
    df["Developer_Name"].isna(),
    df["Developer_Name"].str.upper()
)

In [16]:
df['Developer_Name'].value_counts

<bound method IndexOpsMixin.value_counts of 0                  RMZ
1          PURAVANKARA
2         TATA HOUSING
3              EMBASSY
4              SNN RAJ
              ...     
100995         EMBASSY
100996         BRIGADE
100997             RMZ
100998         EMBASSY
100999      L&T REALTY
Name: Developer_Name, Length: 81884, dtype: str>

In [17]:
df["Price_per_Sqft"] = (df["Ticket_Price_Cr"] * 10_000_000) / df["Unit_Size_Sqft"]  ## price per square feet

In [18]:
df['Price_per_Sqft'].value_counts ## price per square feet

<bound method IndexOpsMixin.value_counts of 0         31679.120594
1         28284.985887
2         13646.976013
3         15175.012103
4         21471.096187
              ...     
100995    13258.612265
100996    31776.329690
100997    23839.912923
100998    14679.430230
100999    41236.793039
Name: Price_per_Sqft, Length: 81884, dtype: float64>

In [19]:
df["Purchase_Quarter"] = pd.to_datetime(
    df["Purchase_Quarter"],
    errors="coerce"
)

In [20]:
df["Quarter_Number"] = df["Purchase_Quarter"].dt.quarter

In [21]:
df['Quarter_Number'].value_counts

<bound method IndexOpsMixin.value_counts of 0         1
1         2
2         4
3         1
4         4
         ..
100995    4
100996    3
100997    4
100998    2
100999    4
Name: Quarter_Number, Length: 81884, dtype: int32>

In [22]:
df["Booking_Flag"] = df["Transaction_Type"].map({
    "Primary": 1,
    "Secondary": 0
})

In [23]:
df['Booking_Flag'].value_counts

<bound method IndexOpsMixin.value_counts of 0         1
1         1
2         1
3         1
4         0
         ..
100995    0
100996    1
100997    1
100998    0
100999    1
Name: Booking_Flag, Length: 81884, dtype: int64>